In [32]:
import pandas as pd

# Peak Memory Usage during AdamW
## Steady State Memory Usage
### Parameters
- input embedding: dv
- L transfomer block:
    - 2 RMSNorm: each has d
    - multi-head attention:
        - Q/K/V/O projection: each has d^2
    - feed-forward SwiGLU: 3 matrices each of which has 4d^2
- final RMSNorm: d
- output embedding: dv

So total parameters = P = L(2d+4d^2+12d^2) + d + dv = Ld(2+16d) + d + 2dv
### Optimizer State
- first and second moment of each parameter's gradient
So total optimizer state = 2P
### Gradients
- one gradient per parameter
So total gradients = P
## Peak Memory Usage
### Activations
- input embedding: bsd
- L transformer block:
    - 2 RMSNorm: each has bsd
    - multi-head attention:
        - Q/K/V/O project: each has bsd
        - attention weights, one before and one after softmax: bhss where h = num_heads
        - weighted sum of V: bsd
    - feed-forward SwiGLU: three 4bsd and one bsd
- final RMSNorm: bsd
- output embedding: bsv

So total activations = A = 2bsd + bsv + L(2bsd+4bsd+2bhss+bsd+13bsd) = bs(2d+v) + Lbs(20d+2hs)
## Summary
So total floats = 4P+A

# GPT2 XL

In [12]:
def get_num_parameters(L, d, v):
    return L * d * (2 + 16 * d) + d + 2 * d *v
def get_num_activations(L, d, v, b, s, h):
    res = b * s * (2 * d + v)
    res += L * b * s * (20 * d + 2 * h * s)
    return res

In [18]:
# 2B parameteres
P = get_num_parameters(L=48, d = 1_600, v=50_300)
P/1e9

2.1271952

In [22]:
# 4B activations
A = get_num_activations(L=48, d = 1_600, v=50_300, b = 1, s = 1_024, h = 25)
A/1e9

4.1442304

total bytes = (2+4b) * 4 bytes = 8+16b billion bytes, where b is batch size

If we have 80GB memory, we can afford batch size = 4

# FLOPs in one step of AdamW

The backward pass has twice the FLOPs of the forward pass

In AdamW, we compute first moment, which takes 3P FLOPs.
We compute second moment, which takes another 4P FLOPs.
LR * m1 / sqrt(m2+eps) takes 4P FLOPs.
weight decay takes another P FLOPs.

So one AdamW step takes 3bF+12P FLOPs.

In [24]:
# number of flops in a forward pass
def compute_num_flops(L, d, v, b, s) -> int:
    res = v + L * (4 * d + 2 * s + 3 * 4 * d)
    res *= 2 * s * d * b
    return res

In [26]:
# 4.5T flops
F = compute_num_flops(L=48, d = 1_600, v=50_300, b = 1, s = 1_024)
F / 1e12

4.5134774272

# Wall Clock

In [27]:
mfu = 0.5

In [28]:
advertised = 19.5e12

In [29]:
b = 1_024
steps = 400_000
num_flops = (3 * b * F + 12 * P) * steps

In [30]:
seconds = num_flops / (advertised * mfu)

In [35]:
# 18 years LOL
pd.Timedelta(seconds=seconds).days/365

18.035616438356165